# Phú — WBS 5.0–6.2: FP-Growth, holdout, Weka

**Dữ liệu:** Dùng đúng dữ liệu, one-hot và cấu hình Apriori của Task 4. Mỗi dòng là một quan sát chiến dịch.

**Kiểm chứng:** Tập test chỉ dùng để kiểm tra 5 luật đã chọn từ exploration.

Xem thêm [TASKS_REPORT.md](../reports/fp_growth/TASKS_REPORT.md).


In [1]:
"""Reproduce WBS 5.0-6.2 from the frozen Apriori exploration results.

Run the notebook cells in order. Set WEKA_JAR to rerun installed Weka Apriori.
"""

from __future__ import annotations

import csv
import hashlib
import inspect
import json
from pathlib import Path
import platform
import re
import subprocess
import time

import mlxtend
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
import numpy as np
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
APRIORI = ROOT / "reports" / "apriori"
FP = ROOT / "reports" / "fp_growth"
VALIDATION = ROOT / "reports" / "validation"
WEKA = ROOT / "reports" / "weka"
MIN_COUNT = 100
# Chosen using exploration only: four distinct yes patterns and one no pattern.
HOLDOUT_RULE_IDS = ("R01", "R02", "R04", "R08", "R10")


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def rule_mask(data: pd.DataFrame, items: list[str]) -> pd.Series:
    result = pd.Series(True, index=data.index)
    for item in items:
        name, value = item.split("=", 1)
        if name not in data:
            raise ValueError(f"Missing item attribute: {name}")
        result &= data[name].astype(str).eq(value)
    return result


def make_rule_metrics(rules: pd.DataFrame, n: int, config: str) -> pd.DataFrame:
    rows = []
    for _, r in rules.iterrows():
        lhs = " & ".join(sorted(r["antecedents"]))
        rhs = " & ".join(sorted(r["consequents"]))
        n_ab = round(r["support"] * n)
        n_a = round(r["antecedent support"] * n)
        n_b = round(r["consequent support"] * n)
        support = n_ab / n
        confidence = n_ab / n_a
        lift = confidence / (n_b / n)
        leverage = support - (n_a / n) * (n_b / n)
        assert np.allclose(
            [r["support"], r["confidence"], r["lift"], r["leverage"]],
            [support, confidence, lift, leverage],
            atol=1e-12,
        )
        rows.append(
            {
                "config": config,
                "lhs": lhs,
                "rhs": rhs,
                "lhs_length": len(r["antecedents"]),
                "n_total": n,
                "antecedent_count": n_a,
                "consequent_count": n_b,
                "supporting_observations": n_ab,
                "support": support,
                "confidence": confidence,
                "lift": lift,
                "leverage": leverage,
                "low_count": n_ab < MIN_COUNT,
            }
        )
    return pd.DataFrame(rows)


def compare_tables(left: pd.DataFrame, right: pd.DataFrame, keys: list[str]) -> dict:
    joined = left.merge(right, on=keys, how="outer", suffixes=("_apriori", "_fp"), indicator=True)
    mismatch = joined[joined["_merge"] != "both"]
    metric_mismatch = 0
    if mismatch.empty:
        for col in ("support", "confidence", "lift", "leverage"):
            if f"{col}_apriori" in joined:
                metric_mismatch += int(
                    (~np.isclose(joined[f"{col}_apriori"], joined[f"{col}_fp"], atol=1e-12)).sum()
                )
        for col in ("antecedent_count", "consequent_count", "supporting_observations"):
            if f"{col}_apriori" in joined:
                metric_mismatch += int((joined[f"{col}_apriori"] != joined[f"{col}_fp"]).sum())
    return {
        "only_apriori": int((joined._merge == "left_only").sum()),
        "only_fp": int((joined._merge == "right_only").sum()),
        "metric_mismatches": metric_mismatch,
    }


import os
from IPython.display import display

for directory in (FP, VALIDATION, WEKA):
    directory.mkdir(parents=True, exist_ok=True)
metadata = json.loads((APRIORI / 'run_metadata.json').read_text(encoding='utf-8'))
source = ROOT / 'data/processed/processed_exploration.csv'
assert sha256(source) == metadata['source_sha256']
configs = pd.read_csv(APRIORI / 'configs.csv')
display(configs)


,config,min_support,min_confidence,max_len
0,A,0.01,0.6,3
1,B,0.02,0.6,3
2,C,0.03,0.6,3


## 5.0 — Chạy FP-Growth với ba cấu hình

Dùng đúng cấu hình A/B/C của Apriori: support tối thiểu 1%/2%/3%, confidence tối thiểu 60% và itemset tối đa 3 item. Cell này xuất itemset, luật và log thời gian FP-Growth cho từng cấu hình.


In [2]:
def run_fp(configs: pd.DataFrame, n: int) -> pd.DataFrame:
    onehot = pd.read_csv(APRIORI / "onehot_exploration.csv").astype(bool)
    if len(onehot) != n or not onehot[["y=yes", "y=no"]].sum(axis=1).eq(1).all():
        raise ValueError("Apriori one-hot data does not match exploration")
    source = (
        pd.read_csv(ROOT / "data" / "processed" / "processed_exploration.csv", sep=";")
        .drop(columns="previous")
        .astype(str)
    )
    if len(source) != n or not onehot.sum(axis=1).eq(source.shape[1]).all():
        raise ValueError("One-hot row count or item count differs from the source")
    for column in source:
        for value in source[column].unique():
            item = f"{column}={value}"
            if item not in onehot or not onehot[item].eq(source[column].eq(value)).all():
                raise ValueError(f"One-hot item differs from the source: {item}")
    logs = []
    experiment_logs = []
    for cfg in configs.itertuples():
        apriori_start = time.perf_counter()
        same_run_apriori = apriori(
            onehot,
            min_support=cfg.min_support,
            use_colnames=True,
            max_len=cfg.max_len,
            low_memory=True,
        )
        apriori_seconds = time.perf_counter() - apriori_start
        start = time.perf_counter()
        itemsets = fpgrowth(
            onehot, min_support=cfg.min_support, use_colnames=True, max_len=cfg.max_len
        )
        mining_seconds = time.perf_counter() - start
        itemsets["length"] = itemsets.itemsets.map(len)
        fi_export = itemsets.copy()
        fi_export["itemsets"] = fi_export.itemsets.map(lambda v: " & ".join(sorted(v)))
        fi_export.to_csv(FP / f"frequent_itemsets_{cfg.config}.csv", index=False)

        kwargs = {"metric": "confidence", "min_threshold": cfg.min_confidence}
        if "num_itemsets" in inspect.signature(association_rules).parameters:
            kwargs["num_itemsets"] = n
        raw = association_rules(itemsets.drop(columns="length"), **kwargs)
        target = raw[
            raw.consequents.map(lambda x: x in (frozenset({"y=yes"}), frozenset({"y=no"})))
            & raw.antecedents.map(lambda x: len(x) > 0 and not any(i.startswith("y=") for i in x))
        ]
        fp_rules = make_rule_metrics(target, n, cfg.config)
        fp_rules.to_csv(FP / f"rule_metrics_{cfg.config}.csv", index=False)
        fp_total_seconds = time.perf_counter() - start
        experiment_logs.append(
            dict(
                config=cfg.config,
                min_support=cfg.min_support,
                min_confidence=cfg.min_confidence,
                max_len=cfg.max_len,
                minimum_joint_count=int(np.ceil(cfg.min_support * n)),
                frequent_itemsets=len(fi_export),
                rules_confidence_pass=len(raw),
                target_rules=len(fp_rules),
                target_yes=int((fp_rules["rhs"] == "y=yes").sum()),
                target_no=int((fp_rules["rhs"] == "y=no").sum()),
                low_count_rules=int(fp_rules["low_count"].sum()),
                eligible_rules=int(((fp_rules["lift"] > 1) & (~fp_rules["low_count"])).sum()),
                mining_seconds=mining_seconds,
                total_seconds=fp_total_seconds,
                **{f"itemsets_length_{k}": int((itemsets["length"] == k).sum()) for k in (1, 2, 3)},
            )
        )
        apriori_fi = pd.read_csv(APRIORI / f"frequent_itemsets_{cfg.config}.csv")
        if len(same_run_apriori) != len(apriori_fi):
            raise AssertionError(f"Apriori rerun differs for {cfg.config}")
        apriori_rules = pd.read_csv(APRIORI / f"rule_metrics_{cfg.config}.csv")
        fi_check = compare_tables(apriori_fi, fi_export, ["itemsets", "length"])
        rule_check = compare_tables(apriori_rules, fp_rules, ["lhs", "rhs"])
        log = dict(
            config=cfg.config,
            min_support=cfg.min_support,
            min_confidence=cfg.min_confidence,
            max_len=cfg.max_len,
            apriori_itemsets=len(apriori_fi),
            fp_itemsets=len(fi_export),
            apriori_target_rules=len(apriori_rules),
            fp_target_rules=len(fp_rules),
            **{f"itemsets_{k}": v for k, v in fi_check.items()},
            **{f"rules_{k}": v for k, v in rule_check.items()},
            apriori_same_run_seconds=apriori_seconds,
            fp_mining_seconds=mining_seconds,
            fp_total_seconds=fp_total_seconds,
        )
        if any(
            v
            for k, v in log.items()
            if k.startswith(("itemsets_", "rules_"))
            and k.endswith(("only_apriori", "only_fp", "metric_mismatches"))
        ):
            raise AssertionError(f"Apriori/FP mismatch in {cfg.config}: {log}")
        logs.append(log)
    result = pd.DataFrame(logs)
    result.to_csv(FP / "comparison.csv", index=False)
    pd.DataFrame(experiment_logs).to_csv(FP / "experiment_logs.csv", index=False)
    return result


configs.to_csv(FP / 'configs.csv', index=False)
comparison = run_fp(configs, metadata['n_observations'])
fp_experiment_logs = pd.read_csv(FP / 'experiment_logs.csv')
display(fp_experiment_logs)


,config,min_support,min_confidence,max_len,minimum_joint_count,frequent_itemsets,rules_confidence_pass,target_rules,target_yes,target_no,low_count_rules,eligible_rules,mining_seconds,total_seconds,itemsets_length_1,itemsets_length_2,itemsets_length_3
0,A,0.01,0.6,3,362,6696,7264,796,9,787,0,469,1.943881,2.145070,56,885,5755
1,B,0.02,0.6,3,724,4483,5684,619,4,615,0,377,1.352981,1.461734,52,680,3751
2,C,0.03,0.6,3,1086,3418,4663,505,0,505,0,316,1.223616,1.386329,49,563,2806


## 5.1 — So sánh FP-Growth và Apriori

Đối chiếu số itemset, số luật hướng đến `y`, các chỉ số và thời gian đo trên cùng máy. Thời gian chỉ là một lần đo; không dùng để kết luận thuật toán nào luôn nhanh hơn.


In [3]:
comparison_columns = ['config', 'apriori_itemsets', 'fp_itemsets', 'apriori_target_rules', 'fp_target_rules', 'itemsets_only_apriori', 'itemsets_only_fp', 'itemsets_metric_mismatches', 'rules_only_apriori', 'rules_only_fp', 'rules_metric_mismatches', 'apriori_same_run_seconds', 'fp_mining_seconds']
display(comparison[comparison_columns])
difference_columns = [c for c in comparison if c.endswith(('only_apriori', 'only_fp', 'metric_mismatches'))]
assert not comparison[difference_columns].to_numpy().any()


,config,apriori_itemsets,fp_itemsets,apriori_target_rules,fp_target_rules,itemsets_only_apriori,itemsets_only_fp,itemsets_metric_mismatches,rules_only_apriori,rules_only_fp,rules_metric_mismatches,apriori_same_run_seconds,fp_mining_seconds
0,A,6696,6696,796,796,0,0,0,0,0,0,0.370291,1.943881
1,B,4483,4483,619,619,0,0,0,0,0,0,0.310602,1.352981
2,C,3418,3418,505,505,0,0,0,0,0,0,0.297381,1.223616


## 5.2 — Kiểm tra chỉ số luật và chọn kết quả cuối

Gộp luật từ ba cấu hình, bỏ trùng, chỉ giữ luật có lift > 1 và ít nhất 100 quan sát hỗ trợ, rồi xếp theo lift, confidence và số quan sát. Kiểm tra 10 luật FP-Growth cuối với bảng Apriori đã cố định.


In [4]:
fp_all = pd.concat([pd.read_csv(FP / f'rule_metrics_{name}.csv') for name in configs['config']], ignore_index=True)
memberships = (fp_all.groupby(['lhs', 'rhs'])['config'].agg(lambda s: ','.join(sorted(s.unique()))).rename('configs').reset_index())
fp_unique = fp_all.drop_duplicates(['lhs', 'rhs']).drop(columns='config').merge(memberships, on=['lhs', 'rhs'])
fp_eligible = fp_unique[(fp_unique['lift'] > 1) & (fp_unique['supporting_observations'] >= 100)].copy()
fp_final = fp_eligible.sort_values(['lift', 'confidence', 'supporting_observations', 'lhs', 'rhs'], ascending=[False, False, False, True, True]).head(10).reset_index(drop=True)
fp_final.insert(0, 'rule_id', [f'R{i:02}' for i in range(1, len(fp_final) + 1)])
apriori_final = pd.read_csv(APRIORI / 'top_10_rules.csv')
assert len(fp_final) == 10
pd.testing.assert_frame_equal(fp_final[['rule_id', 'lhs', 'rhs']], apriori_final[['rule_id', 'lhs', 'rhs']])
for name in ['antecedent_count', 'consequent_count', 'supporting_observations']:
    assert fp_final[name].equals(apriori_final[name]), name
for name in ['support', 'confidence', 'lift', 'leverage']:
    assert np.allclose(fp_final[name], apriori_final[name], atol=1e-12), name
fp_final.to_csv(FP / 'final_top_10_rules.csv', index=False)
display(fp_final[['rule_id', 'lhs', 'rhs', 'supporting_observations', 'support', 'confidence', 'lift', 'leverage', 'configs']])


,rule_id,lhs,rhs,supporting_observations,support,confidence,lift,leverage,configs
0,R01,housing=no & poutcome=success,y=yes,582,0.016092,0.699519,5.979724,0.013401,A
1,R02,campaign_group=1_call & poutcome=success,y=yes,458,0.012663,0.675516,5.774538,0.010470,A
2,R03,marital=married & poutcome=success,y=yes,430,0.011889,0.652504,5.577820,0.009757,A
3,R04,loan=no & poutcome=success,y=yes,743,0.020543,0.651183,5.566531,0.016853,"A,B"
4,R05,contact=cellular & poutcome=success,y=yes,715,0.019769,0.647645,5.536285,0.016198,A
5,R06,default=no & poutcome=success,y=yes,779,0.021538,0.647010,5.530857,0.017644,"A,B"
6,R07,contacted_before=yes & poutcome=success,y=yes,779,0.021538,0.646473,5.526267,0.017641,"A,B"
7,R08,poutcome=success,y=yes,779,0.021538,0.646473,5.526267,0.017641,"A,B"
8,R09,age_group=middle_age & poutcome=success,y=yes,514,0.014211,0.627595,5.364888,0.011562,A
9,R10,contact=unknown & marital=married,y=no,6200,0.171422,0.968447,1.096747,0.015122,"A,B,C"


## 6.0 — Kiểm chứng 5 luật trên test

R01, R02, R04, R08, R10 được chọn theo exploration trước khi đọc test. Tính support, confidence, lift, leverage bằng số đếm trực tiếp, không sinh hoặc chọn lại luật trên test.


In [5]:
def run_holdout() -> pd.DataFrame:
    test = pd.read_csv(ROOT / "data" / "processed" / "processed_test.csv", sep=";")
    top = pd.read_csv(APRIORI / "top_10_rules.csv").set_index("rule_id")
    if len(test) != 9043 or not {"yes", "no"}.issubset(set(test.y)):
        raise ValueError("Unexpected holdout data")
    rows = []
    for rule_id in HOLDOUT_RULE_IDS:
        r = top.loc[rule_id]
        a = rule_mask(test, r.lhs.split(" & "))
        b = rule_mask(test, [r.rhs])
        n, n_a, n_b, n_ab = len(test), int(a.sum()), int(b.sum()), int((a & b).sum())
        if n_a == 0:
            raise ValueError(f"No antecedent observations in holdout for {rule_id}")
        support = n_ab / n
        confidence = n_ab / n_a
        lift = confidence / (n_b / n)
        leverage = support - (n_a / n) * (n_b / n)
        rows.append(
            {
                "rule_id": rule_id,
                "lhs": r.lhs,
                "rhs": r.rhs,
                "exploration_n": int(r.n_total),
                "test_n": n,
                "exploration_joint_count": int(r.supporting_observations),
                "test_antecedent_count": n_a,
                "test_consequent_count": n_b,
                "test_joint_count": n_ab,
                "exploration_support": r.support,
                "test_support": support,
                "exploration_confidence": r.confidence,
                "test_confidence": confidence,
                "confidence_delta": confidence - r.confidence,
                "exploration_lift": r.lift,
                "test_lift": lift,
                "exploration_leverage": r.leverage,
                "test_leverage": leverage,
                "test_rhs_base_rate": n_b / n,
                "test_lift_gt_1": lift > 1,
                "test_confidence_ge_0_6": confidence >= 0.6,
            }
        )
    result = pd.DataFrame(rows)
    result.to_csv(VALIDATION / "holdout_5_rules.csv", index=False)
    return result


holdout = run_holdout()
display(holdout[['rule_id','lhs','rhs','test_antecedent_count',
                 'test_consequent_count','test_joint_count',
                 'exploration_confidence','test_confidence',
                 'exploration_lift','test_lift','test_leverage']])


,rule_id,lhs,rhs,test_antecedent_count,test_consequent_count,test_joint_count,exploration_confidence,test_confidence,exploration_lift,test_lift,test_leverage
0,R01,housing=no & poutcome=success,y=yes,208,1058,147,0.699519,0.706731,5.979724,6.040611,0.013565
1,R02,campaign_group=1_call & poutcome=success,y=yes,159,1058,99,0.675516,0.622642,5.774538,5.321878,0.008891
2,R04,loan=no & poutcome=success,y=yes,288,1058,191,0.651183,0.663194,5.566531,5.668495,0.017395
3,R08,poutcome=success,y=yes,306,1058,199,0.646473,0.650327,5.526267,5.558512,0.018047
4,R10,contact=unknown & marital=married,y=no,1629,7985,1565,0.968447,0.960712,1.096747,1.088005,0.013998


## 6.1 — ARFF và chạy Weka Apriori

ARFF đầy đủ có 14 thuộc tính giống Python. ARFF chiếu giữ `housing`, `campaign_group`, `poutcome`, `y` trên toàn bộ 36.168 dòng để so sánh trực tiếp các luật ngắn R01/R02/R08. Cả hai dùng `-A -c last -N 10000 -T 0 -C 0.6 -U 1 -M 0.01 -D 0.99`. Đặt biến môi trường `WEKA_JAR` trỏ đến file `weka.jar` trước khi chạy notebook nếu muốn tạo lại log Weka.


In [6]:
def arff_quote(value: str) -> str:
    # All current values contain only simple ASCII letters, digits and punctuation.
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", value):
        raise ValueError(f"Unexpected ARFF value: {value!r}")
    return f"'{value}'"


def write_arff(columns: list[str] | None = None) -> Path:
    source = ROOT / "data" / "processed" / "processed_exploration.csv"
    data = pd.read_csv(source, sep=";").drop(columns="previous")
    if columns is not None:
        data = data[columns]
    data = data[[c for c in data if c != "y"] + ["y"]]
    if data.isna().any().any() or len(data) != 36168:
        raise ValueError("Unexpected exploration data")
    path = WEKA / ("exploration_14_items.arff" if columns is None else "exploration_4_items.arff")
    with path.open("w", encoding="ascii", newline="\n") as f:
        f.write("% One row = one observation; same 14 attribute=value items as Python.\n")
        f.write("% previous and duration excluded; y is last for class association rules.\n")
        if columns is not None:
            f.write("% Projection for direct comparison of selected short Python rules.\n")
        f.write("@RELATION bank_marketing_exploration\n\n")
        for col in data:
            values = sorted(data[col].astype(str).unique())
            f.write(f"@ATTRIBUTE {arff_quote(col)} {{{','.join(map(arff_quote, values))}}}\n")
        f.write("\n@DATA\n")
        for row in data.itertuples(index=False, name=None):
            f.write(",".join(arff_quote(str(v)) for v in row) + "\n")
    return path


def run_weka(jar: Path, arff: Path) -> Path:
    # -A mines class association rules, with y as the last ARFF attribute.
    # Weka's upper support bound must exceed the lower bound: equal bounds
    # retain only itemsets at exactly that support count.
    label = "full" if "14_items" in arff.name else "projection"
    output = WEKA / f"weka_apriori_A_{label}.txt"
    command = [
        "java",
        "--add-opens=java.base/java.lang=ALL-UNNAMED",
        "-Xmx4g",
        "-cp",
        str(jar),
        "weka.associations.Apriori",
        "-t",
        str(arff),
        "-c",
        "last",
        "-A",
        "-N",
        "10000",
        "-T",
        "0",
        "-C",
        "0.6",
        "-U",
        "1",
        "-M",
        "0.01",
        "-D",
        "0.99",
    ]
    start = time.perf_counter()
    completed = subprocess.run(
        command,
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=1200,
    )
    output.write_text(completed.stdout + "\nSTDERR:\n" + completed.stderr, encoding="utf-8")
    version = subprocess.run(
        ["java", "-cp", str(jar), "weka.core.Version"], capture_output=True, text=True, check=True
    ).stdout.splitlines()[0]
    (WEKA / f"run_metadata_{label}.json").write_text(
        json.dumps(
            {
                "weka_jar_version": version,
                "jar_sha256": sha256(jar),
                "arff_sha256": sha256(arff),
                "exploration_sha256": sha256(
                    ROOT / "data" / "processed" / "processed_exploration.csv"
                ),
                "command_portable": [
                    "java",
                    "--add-opens=java.base/java.lang=ALL-UNNAMED",
                    "-Xmx4g",
                    "-cp",
                    "<WEKA_JAR>",
                    "weka.associations.Apriori",
                    "-t",
                    arff.name,
                    "-c",
                    "last",
                    "-A",
                    "-N",
                    "10000",
                    "-T",
                    "0",
                    "-C",
                    "0.6",
                    "-U",
                    "1",
                    "-M",
                    "0.01",
                    "-D",
                    "0.99",
                ],
                "runtime_seconds": time.perf_counter() - start,
                "returncode": completed.returncode,
                "java_version": subprocess.run(
                    ["java", "-version"], check=True, capture_output=True, text=True
                ).stderr.strip(),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    if completed.returncode:
        raise RuntimeError(f"Weka failed with exit code {completed.returncode}; see {output}")
    return output


full_arff = write_arff()
projection_arff = write_arff(['housing','campaign_group','poutcome','y'])
print(full_arff.relative_to(ROOT), sha256(full_arff))
print(projection_arff.relative_to(ROOT), sha256(projection_arff))
weka_jar = os.environ.get('WEKA_JAR')
if not weka_jar and os.environ.get('ProgramFiles'):
    installed_jars = sorted(Path(os.environ['ProgramFiles']).glob('Weka-*/weka.jar'))
    if installed_jars:
        weka_jar = str(installed_jars[-1])
if weka_jar:
    print(run_weka(Path(weka_jar), full_arff).relative_to(ROOT))
    print(run_weka(Path(weka_jar), projection_arff).relative_to(ROOT))
else:
    print('Không tìm thấy Weka; dùng log đã lưu. Đặt WEKA_JAR để chạy lại.')


reports\weka\exploration_14_items.arff 5540b7bc2b4c10bdf1348e681d0cce8d9b629a9c57aeb2330c7563dd471dc469
reports\weka\exploration_4_items.arff 02cfd61a3fa187702af54fec136e99f24449b8a4c6404af05ecb48cc74c1a725


reports\weka\weka_apriori_A_full.txt


reports\weka\weka_apriori_A_projection.txt


### Ảnh Weka Explorer: dữ liệu đầy đủ 14 thuộc tính

Ảnh Preprocess xác nhận Weka đọc 36.168 dòng và 14 thuộc tính. Ảnh Associate cho thấy tham số Apriori ở thanh trên và các luật đầu danh sách khi chạy trên dữ liệu đầy đủ. Weka xếp luật theo confidence nên các luật `y=no` xuất hiện trước.

![Weka Preprocess với 14 thuộc tính](../reports/weka/weka_gui_14_data.png)

![Weka Associate trên 14 thuộc tính](../reports/weka/weka_gui_14_rules.png)


### Ảnh Weka Explorer: dữ liệu chiếu 4 thuộc tính

Ảnh Preprocess xác nhận vẫn là 36.168 dòng, chỉ còn `housing`, `campaign_group`, `poutcome`, `y`. Ảnh Associate hiển thị ba luật số 39–41 tương ứng R01, R02, R08 trong Python: `832 ⇒ 582`, `678 ⇒ 458`, `1205 ⇒ 779`. Weka làm tròn confidence thành `0.70`, `0.68`, `0.65`; bảng 6.2 đối chiếu các giá trị này với Python.

![Weka Preprocess với 4 thuộc tính](../reports/weka/weka_gui_4_data.png)

![Ba luật Weka số 39 đến 41](../reports/weka/weka_gui_4_rules.png)


## 6.2 — Đối chiếu 3 luật Weka/Python

Đối chiếu R01/R02/R08 theo năm số liệu ở mỗi bên: số dòng thỏa vế trái, số dòng thỏa cả luật, support, confidence và lift. Weka in trực tiếp hai số đếm và confidence (làm tròn hai chữ số); support và lift phía Weka được tính từ các số đếm đó và phân bố `y` trong chính ARFF đã đưa vào Weka.


In [2]:
def compare_weka_rules() -> pd.DataFrame:
    """Match three frozen Apriori rules to actual Weka projection output."""
    output = WEKA / "weka_apriori_A_projection.txt"
    metadata_path = WEKA / "run_metadata_projection.json"
    if not output.exists() or not metadata_path.exists():
        raise FileNotFoundError("Run Weka on the projection before comparing rules")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata["returncode"] != 0 or metadata["arff_sha256"] != sha256(
        WEKA / "exploration_4_items.arff"
    ):
        raise ValueError("Weka output is stale or did not finish successfully")
    arff = WEKA / "exploration_4_items.arff"
    data_lines = arff.read_text(encoding="ascii").split("@DATA\n", 1)[1].splitlines()
    arff_rows = list(csv.reader((line for line in data_lines if line and not line.startswith("%")), quotechar="'"))
    if len(arff_rows) != 36168 or any(len(row) != 4 for row in arff_rows):
        raise ValueError("Unexpected Weka ARFF rows")
    class_counts = {value: sum(row[-1] == value for row in arff_rows) for value in ("yes", "no")}
    parsed = {}
    pattern = re.compile(
        r"^\s*(\d+)\.\s+(.*?)\s+(\d+)\s+==>\s+(y=(?:yes|no))\s+"
        r"(\d+)\s+conf:\(([^)]+)\)"
    )
    for line in output.read_text(encoding="utf-8-sig").splitlines():
        match = pattern.match(line)
        if match:
            rank, lhs, n_a, rhs, n_ab, confidence = match.groups()
            parsed[(" & ".join(sorted(lhs.split())), rhs)] = (
                int(rank),
                int(n_a),
                int(n_ab),
                float(confidence),
                line.strip(),
            )
    top = pd.read_csv(APRIORI / "top_10_rules.csv").set_index("rule_id")
    rows = []
    for rule_id in ("R01", "R02", "R08"):
        r = top.loc[rule_id]
        key = (r.lhs, r.rhs)
        if key not in parsed:
            raise AssertionError(f"Weka did not report {rule_id}: {key}")
        rank, n_a, n_ab, displayed_confidence, source_line = parsed[key]
        n_weka = len(arff_rows)
        n_b = class_counts[r.rhs.split("=", 1)[1]]
        weka_support = n_ab / n_weka
        weka_confidence_exact = n_ab / n_a
        weka_lift = weka_confidence_exact / (n_b / n_weka)
        if (n_weka, n_b) != (r.n_total, r.consequent_count):
            raise AssertionError(f"Weka ARFF counts differ for {rule_id}")
        if (n_a, n_ab) != (r.antecedent_count, r.supporting_observations):
            raise AssertionError(f"Weka counts differ for {rule_id}")
        if abs(displayed_confidence - r.confidence) > 0.005000001:
            raise AssertionError(f"Weka confidence differs for {rule_id}")
        if not np.allclose([weka_support, weka_confidence_exact, weka_lift], [r.support, r.confidence, r.lift], atol=1e-12):
            raise AssertionError(f"Weka-derived metrics differ for {rule_id}")
        rows.append(
            {
                "rule_id": rule_id,
                "lhs": r.lhs,
                "rhs": r.rhs,
                "weka_rank": rank,
                "weka_antecedent_count": n_a,
                "weka_joint_count": n_ab,
                "weka_support": weka_support,
                "weka_displayed_confidence": displayed_confidence,
                "weka_lift": weka_lift,
                "python_antecedent_count": int(r.antecedent_count),
                "python_joint_count": int(r.supporting_observations),
                "python_support": r.support,
                "python_confidence": r.confidence,
                "python_lift": r.lift,
                "python_leverage": r.leverage,
                "count_match": True,
                "confidence_match_after_rounding": True,
                "weka_source_line": source_line,
            }
        )
    result = pd.DataFrame(rows)
    result.to_csv(WEKA / "three_rule_comparison.csv", index=False)
    return result


weka_check = compare_weka_rules()
metrics_to_compare = [
    ('Antecedent count', 'weka_antecedent_count', 'python_antecedent_count'),
    ('Joint count', 'weka_joint_count', 'python_joint_count'),
    ('Support', 'weka_support', 'python_support'),
    ('Confidence', 'weka_displayed_confidence', 'python_confidence'),
    ('Lift', 'weka_lift', 'python_lift'),
]
comparison_rows = []
for rule in weka_check.itertuples():
    for metric, weka_column, python_column in metrics_to_compare:
        weka_value = getattr(rule, weka_column)
        python_value = getattr(rule, python_column)
        if metric in ('Antecedent count', 'Joint count'):
            weka_text, python_text = str(int(weka_value)), str(int(python_value))
        elif metric == 'Confidence':
            weka_text, python_text = f'{weka_value:.2f}', f'{python_value:.6f}'
        else:
            weka_text, python_text = f'{weka_value:.6f}', f'{python_value:.6f}'
        comparison_rows.append({
            'Rule': rule.rule_id, 'Metric': metric,
            'Weka': weka_text, 'Python': python_text,
            'Result': 'Match after rounding' if metric == 'Confidence' else 'Match',
        })
comparison_view = pd.DataFrame(comparison_rows).set_index(['Rule', 'Metric'])
display(comparison_view)
print('All 3 rules: counts, support, and lift match; Weka confidence matches after rounding.')


Weka    Python                Result
Rule Metric                                                    
R01  Antecedent count       832       832                 Match
     Joint count            582       582                 Match
     Support           0.016092  0.016092                 Match
     Confidence            0.70  0.699519  Match after rounding
     Lift              5.979724  5.979724                 Match
R02  Antecedent count       678       678                 Match
     Joint count            458       458                 Match
     Support           0.012663  0.012663                 Match
     Confidence            0.68  0.675516  Match after rounding
     Lift              5.774538  5.774538                 Match
R08  Antecedent count      1205      1205                 Match
     Joint count            779       779                 Match
     Support           0.021538  0.021538                 Match
     Confidence            0.65  0.646473  Match after rounding
     Lift              5.526267  5.526267                 Match

All 3 rules: counts, support, and lift match; Weka confidence matches after rounding.


## Diễn giải

## Kết quả là mối liên hệ mô tả. Tách ngẫu nhiên phân tầng chưa kiểm tra theo thời gian và không có mã khách hàng để bảo đảm tính độc lập. Các luật `y=yes` ở đây đều chứa `poutcome=success`, nên không thể xem như nhiều giả thuyết độc lập hoặc kết luận nhân quả. Xem báo cáo để có bảng kết quả và giới hạn chi tiết.
